# 🚀 vLLM on Google Colab (T4対応版)

このノートブックでは **vLLM** を使って LLM を高速推論します。

### 前提条件
- ランタイム → ランタイムのタイプを変更 → **GPU (T4)** を選択

### T4での注意点
| 問題 | 対処 |
|---|---|
| vLLM 0.7以降は sm_75(T4) 非対応 | `vllm==0.6.6` を使用 |
| numpy バイナリ非互換 | `numpy==1.26.4` を指定 |
| CUDAグラフコンパイルエラー | `enforce_eager=True` を指定 |
| VRAM不足 | `gpu_memory_utilization=0.5` に設定 |

## ① GPU の確認

In [ ]:
!nvidia-smi

## ② vLLM のインストール

> ⏳ 数分かかります。インストール後は必ず **ランタイム → セッションを再起動** してください。

- `vllm==0.6.6` : T4(sm_75)で動作する最新の安定バージョン
- `numpy==1.26.4` : vLLM 0.6.6 との互換バージョン
- `protobuf==4.25.3` : Colab の protobuf 競合を回避

In [1]:
!pip install vllm==0.6.6 numpy==1.26.4 protobuf==4.25.3 -q
print("✅ インストール完了！ランタイムを再起動してください。")
print("   メニュー: ランタイム → セッションを再起動")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.1/201.1 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 106.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.6/294.6 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.6/87.6 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 906.4/906.4 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13

> ⚠️ **上のセルを実行したら、必ずランタイムを再起動してから続けてください。**

## ③ インストール確認

In [1]:
import vllm
import numpy as np
print(f"vLLM バージョン : {vllm.__version__}")   # → 0.6.6
print(f"NumPy バージョン: {np.__version__}")      # → 1.26.4

vLLM バージョン : 0.6.6
NumPy バージョン: 1.26.4


## ④ モデルのロードと推論

| モデル例 | VRAM目安 | 備考 |
|---|---|---|
| `facebook/opt-125m` | ~1GB | テスト用・軽量 |
| `TinyLlama/TinyLlama-1.1B-Chat-v1.0` | ~3GB | チャット向き |
| `microsoft/phi-2` | ~6GB | 高品質・小型 |
| `mistralai/Mistral-7B-Instruct-v0.2` | ~14GB | T4ギリギリ |

> ⚠️ このセル（`LLM()`）を実行すると GPU メモリを消費します。
> APIサーバー（⑥）と**同時には使わない**でください。

In [2]:
from vllm import LLM, SamplingParams

model_name = "facebook/opt-125m"  # ← 変更する場合はここを編集

print(f"モデルをロード中: {model_name}")
llm = LLM(
    model=model_name,
    dtype="float16",           # T4向け
    enforce_eager=True,        # CUDAグラフを無効化（T4必須）
    gpu_memory_utilization=0.5 # VRAM の50%を使用
)
print("✅ モデルのロード完了！")

モデルをロード中: facebook/opt-125m


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/651 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


INFO 03-13 23:19:57 config.py:510] This model supports multiple tasks: {'reward', 'classify', 'generate', 'embed', 'score'}. Defaulting to 'generate'.
WARNING 03-13 23:19:57 cuda.py:98] To see benefits of async output processing, enable CUDA graph. Since, enforce-eager is enabled, async output processor cannot be used
WARNING 03-13 23:19:57 config.py:642] Async output processing is not supported on the current platform type cuda.
INFO 03-13 23:19:57 llm_engine.py:234] Initializing an LLM engine (v0.6.6) with config: model='facebook/opt-125m', speculative_config=None, tokenizer='facebook/opt-125m', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=2048, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=True, kv_cache_dtype=auto, quantization_param_path

tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/441 [00:00<?, ?B/s]

AttributeError: GPT2Tokenizer has no attribute all_special_tokens_extended

In [ ]:
sampling_params = SamplingParams(
    temperature=0.8,
    top_p=0.95,
    max_tokens=100,
)

prompts = [
    "The future of artificial intelligence is",
    "Once upon a time in a distant galaxy,",
]

outputs = llm.generate(prompts, sampling_params)

print("=" * 60)
for i, output in enumerate(outputs):
    print(f"\n📝 プロンプト {i+1}: {output.prompt}")
    print(f"💬 生成結果  : {output.outputs[0].text}")
    print("-" * 60)

## ⑤ インタラクティブな推論（オプション）

In [ ]:
user_prompt = input("プロンプトを入力してください: ")

if user_prompt.strip():
    params = SamplingParams(temperature=0.7, max_tokens=200)
    result = llm.generate([user_prompt], params)
    print("\n💬 生成結果:")
    print(result[0].outputs[0].text)
else:
    print("⚠️ プロンプトが空です")

## ⑥ OpenAI 互換サーバーの起動（オプション）

> ⚠️ **④のセル（`LLM()`）を実行済みの場合は、先にランタイムを再起動してください。**  
> サーバーと `LLM()` を同時に動かすと VRAM が不足します。

In [ ]:
import subprocess

server_model = "facebook/opt-125m"  # ← モデルを変更する場合はここを編集

proc = subprocess.Popen(
    ["python", "-m", "vllm.entrypoints.openai.api_server",
     "--model", server_model,
     "--host", "0.0.0.0",
     "--port", "8000",
     "--dtype", "float16",
     "--enforce-eager",
     "--gpu-memory-utilization", "0.5"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

print("サーバー起動中... ログを監視します")
for _ in range(120):
    line = proc.stdout.readline().decode("utf-8", errors="ignore")
    if line:
        print(line, end="")
    if "Application startup complete" in line or "Uvicorn running" in line:
        print("\n✅ サーバー起動完了！")
        break

In [ ]:
# curl でテスト
!curl http://localhost:8000/v1/completions \
  -H "Content-Type: application/json" \
  -d '{"model": "facebook/opt-125m", "prompt": "Hello, my name is", "max_tokens": 50}'

In [ ]:
# openai ライブラリ経由でも呼び出せます
!pip install openai -q

from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="dummy",  # vLLM はキー不要なのでダミーでOK
)

response = client.completions.create(
    model="facebook/opt-125m",
    prompt="The future of AI is",
    max_tokens=50,
)

print(response.choices[0].text)

---

## 📌 トラブルシューティング

| エラー | 原因 | 対処 |
|---|---|---|
| `numpy.dtype size changed` | numpy バイナリ非互換 | `numpy==1.26.4` を指定してランタイム再起動 |
| `Free memory ... less than desired` | VRAM 不足 | `LLM()` を先に終了 or ランタイム再起動 |
| `-arch=compute_ is unsupported` | vLLM 0.7+ が T4 非対応 | `vllm==0.6.6` を使用 |
| `Connection refused` | サーバー起動前に curl | ログで `startup complete` を確認してから実行 |
| `MessageFactory` エラー | protobuf バージョン競合 | 無視してOK（動作に影響なし）|